In [0]:
%sql
DECLARE OR REPLACE VARIABLE myusername STRING; 

SET VAR myusername = (select concat( 'gold_dimensional_', replace(replace(current_user(),'@',''),'.','')));
--SET VAR myusername = 'gold';

In [0]:
%sql
use catalog skofy27

In [0]:
# Get all tables in skofy27.gold_dimensional schema
tables = spark.sql("SHOW TABLES IN IDENTIFIER(myusername)").collect()

print(f"Found {len(tables)} tables")
print("\nProcessing tables to enable Uniform (Iceberg compatibility)...\n")

# Track results
success_count = 0
failed_tables = []

# Process each table: disable deletion vectors -> reorg -> enable Uniform
for table in tables:
    table_name = table.tableName
    database = table.database
    full_table_name = f"{database}.{table_name}"
    
    try:
        print(f"Processing: {full_table_name}")
        
        # Step 1: Disable deletion vectors
        print(f"  Step 1: Disabling deletion vectors...")
        spark.sql(f"""
            ALTER TABLE {full_table_name}
            SET TBLPROPERTIES (
                'delta.enableDeletionVectors' = 'false'
            )
        """)
        
        # Step 2: Reorg table to rewrite files
        print(f"  Step 2: Running REORG to rewrite table...")
        spark.sql(f"REORG TABLE {full_table_name} APPLY (PURGE)")
        
        # Step 3: Enable Uniform with Iceberg
        print(f"  Step 3: Enabling Uniform (Iceberg)...")
        spark.sql(f"""
            ALTER TABLE {full_table_name}
            SET TBLPROPERTIES (
                'delta.universalFormat.enabledFormats' = 'iceberg',
                'delta.enableIcebergCompatV2' = 'true',
                'delta.columnMapping.mode' = 'name'
            )
        """)

         # Step 4: Reorg table to rewrite files
        print(f"  Step 2: Running REORG to rewrite table...")
        spark.sql(f"REORG TABLE {full_table_name} APPLY (UPGRADE UNIFORM(ICEBERG_COMPAT_VERSION=2))")
        
        print(f"✓ Successfully enabled Uniform for {full_table_name}\n")
        success_count += 1
    except Exception as e:
        print(f"✗ Failed to process {full_table_name}: {str(e)}\n")
        failed_tables.append((full_table_name, str(e)))

# Summary
print("="*60)
print(f"Summary: {success_count}/{len(tables)} tables successfully processed")
if failed_tables:
    print(f"\nFailed tables ({len(failed_tables)}):")
    for table_name, error in failed_tables:
        print(f"  - {table_name}: {error}")

In [0]:
%sql 
vacuum skofy27.gold_dimensional_joelrolandsnowflakecom.dim_customer